# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [164]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix
from sklearn.model_selection import cross_val_score, ParameterGrid
import itertools
from sklearn.linear_model import LogisticRegression
import joblib

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [165]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
day = pd.read_csv('../data/dayofweek.csv')
df = pd.concat([day['dayofweek'], df], axis=1)
X = df.drop('dayofweek', axis=1)
y = df['dayofweek']
df.head()

,dayofweek,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,uid_user_16,uid_user_17,uid_user_18,uid_user_19,uid_user_2,uid_user_20,uid_user_21,uid_user_22,uid_user_23,uid_user_24,uid_user_25,uid_user_26,uid_user_27,uid_user_28,uid_user_29,uid_user_3,uid_user_30,uid_user_31,uid_user_4,uid_user_6,uid_user_7,uid_user_8,labname_code_rvw,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,4,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,4,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,4,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [166]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=21
)

In [167]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=21
)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [168]:
def score(model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)
    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, average='weighted')
    recall = recall_score(y_valid, y_pred, average='weighted')
    print(f'accuracy is {accuracy:.5f}')
    print(f'precision is {precision:.5f}')
    print(f'recall is {recall:.5f}')

In [169]:
svm = SVC(kernel='rbf', C=10, gamma='auto', class_weight=None, probability=True, random_state=21)
tree = DecisionTreeClassifier(class_weight='balanced', criterion='gini', max_depth=21, random_state=21)
forest = RandomForestClassifier(n_estimators=100, max_depth=24, criterion='gini', class_weight=None, random_state=21)
models = [
    svm,
    tree,
    forest
]
for model in models:
    print(f'For {model.__class__.__name__}')
    score(model)
    print('\n')

For SVC
accuracy is 0.87778
precision is 0.88162
recall is 0.87778


For DecisionTreeClassifier
accuracy is 0.86667
precision is 0.87170
recall is 0.86667


For RandomForestClassifier
accuracy is 0.89259
precision is 0.89287
recall is 0.89259




## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [170]:
eclf = VotingClassifier(
    estimators=[
    ('svc', svm),
    ('tree', tree),
    ('rf', forest)
    ],
    voting='hard'
)
score(eclf)

accuracy is 0.89630
precision is 0.89632
recall is 0.89630


In [171]:
eclf = VotingClassifier(
    estimators=[
    ('svc', svm),
    ('tree', tree),
    ('rf', forest)
    ],
    voting='soft'
)
score(eclf)

accuracy is 0.88519
precision is 0.88813
recall is 0.88519


In [172]:
params = {
    'svc__kernel': ['rbf'],
    'svc__C': [10],
    'svc__gamma': ['auto'],
    'svc__class_weight': [None],

    'tree__max_depth': [21],
    'tree__class_weight': ['balanced'],
    'tree__criterion': ['gini'],

    'rf__n_estimators': [50],
    'rf__max_depth': [47],
    'rf__class_weight': [None],
    'rf__criterion': ['gini'],

    'voting': ['hard', 'soft'],
    'weights': list(itertools.product(range(1, 5), repeat=3))
}

In [173]:
grid = GridSearchCV(eclf, params)

In [174]:
grid.fit(X_train, y_train)

GridSearchCV(estimator=VotingClassifier(estimators=[('svc',
                                                     SVC(C=10, gamma='auto',
                                                         probability=True,
                                                         random_state=21)),
                                                    ('tree',
                                                     DecisionTreeClassifier(class_weight='balanced',
                                                                            max_depth=21,
                                                                            random_state=21)),
                                                    ('rf',
                                                     RandomForestClassifier(max_depth=24,
                                                                            random_state=21))],
                                        voting='soft'),
             param_grid={'rf__class_weight': [None], 'rf__criter

In [175]:
pd.set_option('max_columns', None)

In [176]:
pd.DataFrame(grid.cv_results_).sort_values(by='rank_test_score')

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_rf__class_weight,param_rf__criterion,param_rf__max_depth,param_rf__n_estimators,param_svc__C,param_svc__class_weight,param_svc__gamma,param_svc__kernel,param_tree__class_weight,param_tree__criterion,param_tree__max_depth,param_voting,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
115,0.179987,0.005022,0.007118,0.000305,None,gini,47,50,10,None,auto,rbf,balanced,gini,21,soft,"(4, 1, 4)","{'rf__class_weight': None, 'rf__criterion': 'g...",0.888889,0.888889,0.921296,0.879070,0.902326,0.896094,0.014613,1
99,0.179351,0.004180,0.006990,0.000221,None,gini,47,50,10,None,auto,rbf,balanced,gini,21,soft,"(3, 1, 4)","{'rf__class_weight': None, 'rf__criterion': 'g...",0.888889,0.888889,0.925926,0.874419,0.902326,0.896090,0.017335,2
98,0.179655,0.005094,0.007308,0.000293,None,gini,47,50,10,None,auto,rbf,balanced,gini,21,soft,"(3, 1, 3)","{'rf__class_weight': None, 'rf__criterion': 'g...",0.888889,0.884259,0.921296,0.879070,0.902326,0.895168,0.015176,3
67,0.180507,0.004087,0.007329,0.000183,None,gini,47,50,10,None,auto,rbf,balanced,gini,21,soft,"(1, 1, 4)","{'rf__class_weight': None, 'rf__criterion': 'g...",0.888889,0.879630,0.939815,0.865116,0.902326,0.895155,0.025411,4
82,0.180415,0.004789,0.006833,0.000148,None,gini,47,50,10,None,auto,rbf,balanced,gini,21,soft,"(2, 1, 3)","{'rf__class_weight': None, 'rf__criterion': 'g...",0.893519,0.884259,0.930556,0.869767,0.897674,0.895155,0.020127,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48,0.177887,0.005401,0.007094,0.000324,None,gini,47,50,10,None,auto,rbf,balanced,gini,21,hard,"(4, 1, 1)","{'rf__class_weight': None, 'rf__criterion': 'g...",0.861111,0.833333,0.875000,0.795349,0.846512,0.842261,0.027297,121
12,0.179556,0.004540,0.007353,0.000323,None,gini,47,50,10,None,auto,rbf,balanced,gini,21,hard,"(1, 4, 1)","{'rf__class_weight': None, 'rf__criterion': 'g...",0.810185,0.842593,0.907407,0.827907,0.809302,0.839479,0.036129,125
13,0.179833,0.004902,0.007219,0.000428,None,gini,47,50,10,None,auto,rbf,balanced,gini,21,hard,"(1, 4, 2)","{'rf__class_weight': None, 'rf__criterion': 'g...",0.810185,0.842593,0.907407,0.827907,0.809302,0.839479,0.036129,125
28,0.180430,0.005004,0.007714,0.000747,None,gini,47,50,10,None,auto,rbf,balanced,gini,21,hard,"(2, 4, 1)","{'rf__class_weight': None, 'rf__criterion': 'g...",0.810185,0.842593,0.907407,0.827907,0.809302,0.839479,0.036129,125


In [177]:
eclf2 = VotingClassifier(
    estimators=[
    ('svc', svm),
    ('tree', tree),
    ('rf', forest)
    ],
    voting='soft',
    weights=[4,1,4]
)
score(eclf2)

accuracy is 0.89630
precision is 0.89713
recall is 0.89630


## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [178]:
bc = BaggingClassifier(base_estimator=svm, random_state=21)
grid_search = GridSearchCV(estimator=bc, param_grid={'n_estimators': [10, 25, 50, 100]}, scoring='accuracy', n_jobs=-1)
score(grid_search)

accuracy is 0.88148
precision is 0.89035
recall is 0.88148


In [179]:
print(f"Best params is {grid_search.best_params_}")

Best params is {'n_estimators': 50}


In [180]:
bc = BaggingClassifier(base_estimator=svm, random_state=21).fit(X_train, y_train)
y_pred = bc.predict(X_test)
round(pd.DataFrame({
    'accuracy': [accuracy_score(y_test, y_pred)],
    'precision': [precision_score(y_test, y_pred, average='weighted')],
    'recall': [recall_score(y_test, y_pred, average='weighted')]
}),5)

,accuracy,precision,recall
0,0.86391,0.86966,0.86391


## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [181]:
res = []
for n in range(2, 8):
    for passthrough in [True, False]:
        skf = StratifiedKFold(n_splits=n, random_state=21, shuffle=True)
        stack = StackingClassifier(
            estimators=[('svm', svm), ('tree', tree), ('forest', forest)],
            final_estimator=LogisticRegression(solver='liblinear'),
            cv=skf)
        
        stack.fit(X_train, y_train)
        y_pred = stack.predict(X_valid)
        
        accuracy = accuracy_score(y_valid, y_pred)
        precision = precision_score(y_valid, y_pred, average='weighted')
        recall = recall_score(y_valid, y_pred, average='weighted')
        
        res.append({
            'n_splits': n,
            'passthrough': passthrough,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall
        })
res = pd.DataFrame(res).sort_values(['accuracy', 'precision'], ascending=[False, False]).round(5)
res

,n_splits,passthrough,accuracy,precision,recall
4,4,True,0.90741,0.90970,0.90741
5,4,False,0.90741,0.90970,0.90741
6,5,True,0.90370,0.90482,0.90370
7,5,False,0.90370,0.90482,0.90370
8,6,True,0.90000,0.90083,0.90000
9,6,False,0.90000,0.90083,0.90000
0,2,True,0.89630,0.89842,0.89630
1,2,False,0.89630,0.89842,0.89630
2,3,True,0.89630,0.89800,0.89630
3,3,False,0.89630,0.89800,0.89630


In [182]:
skf = StratifiedKFold(n_splits=4, random_state=21, shuffle=True)
final_model = StackingClassifier(
    estimators=[('svm', svm), ('tree', tree), ('forest', forest)],
    final_estimator=LogisticRegression(solver='liblinear'),
    cv=skf,
    passthrough=True
)

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

print(f"accuracy is {accuracy_score(y_test, y_pred):.5f}")
print(f"precision is {precision_score(y_test, y_pred, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred, average='weighted'):.5f}")

accuracy is 0.90533
precision is 0.90711
recall is 0.90533


## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [183]:
cm = confusion_matrix(y_test, y_pred)
res = []
for day in range(cm.shape[0]):
    total = cm[day].sum()
    error = total - cm[day,day]
    percent = round(error/total*100, 2)
    res.append({
        'total': total,
        'correct': cm[day,day],
        'error': error,
        'percent': percent})
pd.DataFrame(res)
joblib.dump(final_model, filename='StackingClassifier.joblib')

['StackingClassifier.joblib']

In [184]:
df_test = X_test.copy()
df_test['true_weekday'] = y_test.values
df_test['pred_weekday'] = y_pred
df_test['is_error'] = df_test['true_weekday'] != df_test['pred_weekday']

In [185]:
def calculate(desired):
    desired_col = [col for col in df_test.columns if col.startswith(f'{desired}_')]
    res = []
    for col in desired_col:
        true_col = df_test[df_test[col] == 1]
        if len(true_col) > 0:
            total = len(true_col)
            error = true_col['is_error'].sum()
            percent = round(error/total*100, 2)
            res.append({
            'labname': col.replace('labname_', ''),
            'total_samples': total,
            'errors': error,
            'error_percent': percent
        })
    return pd.DataFrame(res)

In [186]:
calculate('uid')

,labname,total_samples,errors,error_percent
0,uid_user_1,9,0,0.00
1,uid_user_10,12,0,0.00
2,uid_user_12,12,0,0.00
3,uid_user_13,17,3,17.65
4,uid_user_14,31,3,9.68
5,uid_user_15,2,0,0.00
6,uid_user_16,5,1,20.00
7,uid_user_17,7,1,14.29
8,uid_user_18,6,1,16.67
9,uid_user_19,19,3,15.79


In [187]:
calculate('labname')

,labname,total_samples,errors,error_percent
0,code_rvw,13,1,7.69
1,lab03,1,1,100.00
2,lab03s,1,0,0.00
3,lab05s,6,1,16.67
4,laba04,35,9,25.71
5,laba04s,25,5,20.00
6,laba05,47,0,0.00
7,laba06,9,1,11.11
8,laba06s,15,2,13.33
9,project1,186,12,6.45
